# 02 · CUDA-Q Backends

The same GHZ circuit runs on every simulator backend: **only the `cudaq.set_target(...)` line changes**.

| Backend | Target / option | Method | Qubits here |
|---|---|---|---|
| CPU state vector | `qpp-cpu` | full state vector, multi-threaded CPU | 30 |
| Single GPU | `nvidia` (`fp32`, `fp64`) | full state vector on one GPU | 30 |
| Multi-GPU | `nvidia`, `option="mgpu"` | one state vector **distributed** over several GPUs (MPI) | 30 |
| Multi-QPU | `nvidia`, `option="mqpu"` | each GPU is an independent QPU | 30 |
| Tensor network | `tensornet` (`fp32`, `fp64`) | exact tensor-network contraction (cuTensorNet) | **90** |
| Matrix product state | `tensornet-mps` (`fp32`, `fp64`) | approximate MPS (cuTensorNet) | **90** |

**Why 30 vs 90 qubits?** A state vector stores $2^n$ amplitudes: 30 qubits is 8 GiB in single precision (16 GiB in double) and every extra qubit doubles it, so 90 qubits is impossible. Tensor-network methods never build the full state; the GHZ circuit has very little entanglement structure, so 90 qubits contract quickly.

**Kernel:** select **Python (CUDA-Q GPU)** in the top-right corner.

## Setup: a GHZ kernel with the qubit count as an argument

In [ ]:
import cudaq
import timeit

print("CUDA-Q version:", cudaq.__version__)
print("GPUs visible  :", cudaq.num_available_gpus())

N_SV = 30   # qubits for state-vector backends
N_TN = 90   # qubits for tensornet / tensornet-mps
SHOTS = 1000


@cudaq.kernel
def ghz_kernel(qubit_count: int):
    # Allocate qubits
    qubits = cudaq.qvector(qubit_count)
    # Create GHZ state
    h(qubits[0])
    for i in range(1, qubit_count):
        x.ctrl(qubits[0], qubits[i])
    # Measure qubits in z-basis
    mz(qubits)


timings = {}

def run_ghz(target, qubit_count, option=None, shots=SHOTS):
    """Set the target, sample the GHZ circuit and record the time."""
    if option:
        cudaq.set_target(target, option=option)
    else:
        cudaq.set_target(target)
    label = target + (f" ({option})" if option else "")
    print(f"Target: {label} | qubits: {qubit_count} | shots: {shots}")
    start = timeit.default_timer()
    result = cudaq.sample(ghz_kernel, qubit_count, shots_count=shots)
    elapsed = timeit.default_timer() - start
    for bits, count in result.items():
        print(f"  {bits} : {count}")
    print(f"  time: {elapsed:.2f} s")
    timings[f"{label}, {qubit_count} qubits"] = elapsed
    return result

Let's draw a small version of the circuit first: one Hadamard followed by a fan of CNOTs from qubit 0.

In [ ]:
print(cudaq.draw(ghz_kernel, 5))

## 1 · CPU state vector: `qpp-cpu`
The multi-threaded CPU simulator (based on Q++). It needs no GPU but is by far the slowest at this size and holds the full $2^{30}$ state vector (16 GiB) in RAM.

> ⏱ Reference: 30 qubits took **292 s** on a full 128-core RM node (`GHZ/OUTPUT/CPU_qpp-cpu_30qubit`). A GPU-shared Jupyter session has far fewer cores, so expect this cell to take much longer. Lower `N_SV` here if you only want to see it work.

In [ ]:
run_ghz("qpp-cpu", N_SV);

## 2 · Single GPU: `nvidia`
The cuStateVec-based GPU state-vector simulator. The default precision is single (`fp32`, 8 GiB for 30 qubits).

> ⏱ Reference: 31 qubits in **1.4 s** on one V100 (`GHZ/OUTPUT/GPU_nvidia_31qubit`).

In [ ]:
run_ghz("nvidia", N_SV);

### `nvidia` with double precision (`fp64`)
Same simulator with complex128 amplitudes: twice the memory (16 GiB for 30 qubits) and higher accuracy.

In [ ]:
run_ghz("nvidia", N_SV, option="fp64");

## 3 · Multi-GPU: `nvidia`, `option="mgpu"`
`mgpu` **distributes one state vector over several GPUs** with MPI, so you can simulate more qubits than fit on a single GPU (each extra doubling of GPUs adds one qubit). Inside a notebook there is a single process, so it runs on one GPU here.

> ⏱ Reference: 32 qubits in **2.2 s** on 2 × V100 with `mpirun -np 2` (`GHZ/OUTPUT/GPU_mgpu_32qubit`).

In [ ]:
run_ghz("nvidia", N_SV, option="mgpu");

To really use several GPUs, request them in the session (e.g. `--gres=gpu:h100-80:2`), save a script and launch it with MPI from the **Terminal**:

```bash
# ghz_mgpu.py contains ghz_kernel + cudaq.set_target("nvidia", option="mgpu") + cudaq.sample(...)
apptainer exec --nv /ocean/containers/cuda-quantum_latest.sif \
    mpirun -np 2 python3 ghz_mgpu.py
```

## 4 · Multi-QPU: `nvidia`, `option="mqpu"`
`mqpu` treats **each GPU as an independent virtual QPU**. Each QPU holds a full copy of its own circuit, so it doesn't let you simulate more qubits; it lets you run **many circuits at once**. Use `sample_async` with a `qpu_id` to dispatch work to each QPU.

In [ ]:
cudaq.set_target("nvidia", option="mqpu")
num_qpus = cudaq.get_target().num_qpus()
print("Number of QPUs:", num_qpus)

start = timeit.default_timer()
futures = [cudaq.sample_async(ghz_kernel, N_SV, shots_count=SHOTS, qpu_id=i)
           for i in range(num_qpus)]
for i, fut in enumerate(futures):
    print(f"QPU {i}: {fut.get()}")
elapsed = timeit.default_timer() - start
print(f"time: {elapsed:.2f} s for {num_qpus} circuit(s)")
timings[f"nvidia (mqpu x{num_qpus}), {N_SV} qubits"] = elapsed

## 5 · Tensor network: `tensornet`
Contracts the circuit as a tensor network with cuTensorNet. The result is **exact**, and memory depends on the circuit's structure rather than $2^n$, so we can use **90 qubits**.

> ⏱ Reference (V100): 90 qubits in **34.1 s** (fp64) and **36.6 s** (fp32); 48 qubits in 8.9 s; 60 qubits in 12.5 s (`GHZ/OUTPUT/GPU_tensornet_*`).

In [ ]:
run_ghz("tensornet", N_TN, option="fp64");

Single precision (`fp32`):

In [ ]:
run_ghz("tensornet", N_TN, option="fp32");

## 6 · Matrix product state: `tensornet-mps`
Represents the state as a **matrix product state** and truncates small singular values, making it approximate but very scalable for low-entanglement circuits. A GHZ state needs a bond dimension of only 2, so MPS handles 90 qubits easily.

In [ ]:
run_ghz("tensornet-mps", N_TN, option="fp64");

Single precision (`fp32`):

In [ ]:
run_ghz("tensornet-mps", N_TN, option="fp32");

## Summary of this session

In [ ]:
print(f"{'backend':<45} {'time (s)':>10}")
print("-" * 56)
for label, sec in timings.items():
    print(f"{label:<45} {sec:>10.2f}")

### Reference timings (V100, from `GHZ/OUTPUT`)

| Backend | Qubits | Time (s) |
|---|---|---|
| `qpp-cpu` (128 cores) | 30 | 292.0 |
| `qpp-cpu` (128 cores) | 31 | 611.6 |
| `nvidia` (1 GPU) | 31 | 1.4 |
| `nvidia` `mgpu` (2 GPUs) | 32 | 2.2 |
| `tensornet` | 48 | 8.9 |
| `tensornet` | 60 | 12.5 |
| `tensornet` fp64 | 90 | 34.1 |
| `tensornet` fp32 | 90 | 36.6 |

**Rules of thumb**
- Up to roughly 30–34 qubits on one or a few GPUs, use `nvidia` (add `mgpu` to scale qubits, `mqpu` to scale throughput).
- For many qubits with limited entanglement, use `tensornet-mps`; for exact results on shallow or structured circuits, use `tensornet`.
- Use `qpp-cpu` for small tests or when no GPU is available.